# Pydantic

## Imports

In [11]:
!pip install -q pydantic-ai

In [2]:
from typing import List

from pydantic import BaseModel, Field
from pydantic_ai import Agent

## 1. Define the structured output

In [3]:
# This is the schema we want the agent to return.
# Instead of free-form text only, the model must produce output
# that matches this structure.
class StudyPlan(BaseModel):
    topic: str = Field(description="The main topic the user wants to learn")
    difficulty: str = Field(description="Beginner, intermediate, or advanced")
    steps: List[str] = Field(description="Ordered learning steps")
    estimated_days: int = Field(description="Estimated number of days to complete")
    summary: str = Field(description="Short summary of the plan")

## 2. Create the agent

In [4]:
# Add OPEN_API_Key
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [5]:
# You can swap the model string depending on your provider.
# Examples:
#   "openai:gpt-5-mini"
#   "google-gla:gemini-2.5-flash"
#
# The instructions act like system prompts / guardrails.
agent = Agent(
    "openai:gpt-5-mini",
    output_type=StudyPlan,
    instructions=(
        "You are a senior software engineer and mentor. "
        "Help beginners learn technical topics in a practical way. "
        "Be concise, accurate, and return a realistic learning plan."
    ),
)

## 3. Add tools

In [6]:
# Tools are Python functions the agent can call.
# The LLM decides when to call them.
#
# This is useful because the model should not hardcode everything itself.
# Instead, it can delegate tasks to deterministic Python functions.

@agent.tool_plain
def estimate_days_for_topic(topic: str, difficulty: str) -> int:
    """
    Estimate how many days a learner might need for a topic.
    This is a simple demo rule-based function.
    """
    topic = topic.lower()
    difficulty = difficulty.lower()

    base_days = {
        "beginner": 7,
        "intermediate": 14,
        "advanced": 21,
    }.get(difficulty, 10)

    # Add a little simple business logic
    if "agent" in topic or "llm" in topic:
        return base_days + 3
    if "kubernetes" in topic or "distributed systems" in topic:
        return base_days + 5

    return base_days


@agent.tool_plain
def get_learning_steps(topic: str) -> List[str]:
    """
    Return a reusable step list for a given topic.
    In a real application this might query a database, internal wiki,
    vector store, or API.
    """
    topic_lower = topic.lower()

    if "pydanticai" in topic_lower:
        return [
            "Understand what an agent is and how prompts drive behavior",
            "Learn how Agent() is initialized with a model and instructions",
            "Add Python tool functions for external actions",
            "Use Pydantic models for structured output validation",
            "Test prompts and inspect where tool calls happen",
            "Add observability, retries, and evaluation later",
        ]

    return [
        "Learn the fundamentals of the topic",
        "Build a small hands-on project",
        "Add error handling and validation",
        "Test edge cases",
        "Document decisions and tradeoffs",
    ]

## 4. Run the agent

In [8]:
# We ask the agent for a study plan.
# The model can choose to use the tools above, then produce a final
# response that matches the StudyPlan schema.
prompt = """
Create a beginner study plan for learning PydanticAI agents.
Set difficulty to beginner.
"""

result = await agent.run(prompt)

## 5. Access the validated output


In [9]:
# Newer PydanticAI versions use .output
# rather than older patterns. The final object here is a StudyPlan instance.
plan = result.output

print("=== Structured Output ===")
print(plan.model_dump_json(indent=2))

=== Structured Output ===
{
  "topic": "PydanticAI agents",
  "difficulty": "beginner",
  "steps": [
    "1. Basics: What is an agent? Read a short introduction to AI agents and the role of prompts. Try a minimal chat example with your preferred LLM (OpenAI/Anthropic).",
    "2. Install & setup: Create a virtual environment and install pydantic-ai (or the current package name), OpenAI/llm SDK, and any extras. Configure API keys securely (env vars or a secrets manager).",
    "3. Initialize an Agent: Use Agent(model=..., instructions='...') to create a basic agent. Experiment with different instructions and system messages.",
    "4. Add Python tools: Write simple tool functions (e.g., get_time, fetch_url, compute_sum) and register them with the agent so it can call external code.",
    "5. Structured outputs with Pydantic: Define Pydantic models for expected outputs, attach them to tools or agent responses, and validate outputs to enforce schema correctness.",
    "6. Prompt engineerin

## 6. Example of accessing fields directly


In [10]:
print("\n=== Friendly View ===")
print("Topic:", plan.topic)
print("Difficulty:", plan.difficulty)
print("Estimated Days:", plan.estimated_days)
print("Steps:")
for i, step in enumerate(plan.steps, start=1):
    print(f"{i}. {step}")
print("Summary:", plan.summary)


=== Friendly View ===
Topic: PydanticAI agents
Difficulty: beginner
Estimated Days: 10
Steps:
1. 1. Basics: What is an agent? Read a short introduction to AI agents and the role of prompts. Try a minimal chat example with your preferred LLM (OpenAI/Anthropic).
2. 2. Install & setup: Create a virtual environment and install pydantic-ai (or the current package name), OpenAI/llm SDK, and any extras. Configure API keys securely (env vars or a secrets manager).
3. 3. Initialize an Agent: Use Agent(model=..., instructions='...') to create a basic agent. Experiment with different instructions and system messages.
4. 4. Add Python tools: Write simple tool functions (e.g., get_time, fetch_url, compute_sum) and register them with the agent so it can call external code.
5. 5. Structured outputs with Pydantic: Define Pydantic models for expected outputs, attach them to tools or agent responses, and validate outputs to enforce schema correctness.
6. 6. Prompt engineering: Practice designing instru